In [1]:
import os
import gc
import re
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing: {DEVICE}")

PyTorch: 2.8.0+cu126
CUDA: False

Using: cpu


In [2]:

COMPETITION_DATA = "/kaggle/input/deep-past-initiative-machine-translation"
MODEL_PATH = "/kaggle/input/akkadian-pretrained-and-finetuned-model/best"
MAX_LENGTH = 192
BATCH_SIZE = 8          
NUM_BEAMS = 5          
LENGTH_PENALTY = 1.0
NO_REPEAT_NGRAM = 3
print("Configuration loaded!")

Configuration loaded!


In [3]:


model_path = None

# Check primary path
if os.path.exists(MODEL_PATH):
    # Check if it has model files
    files = os.listdir(MODEL_PATH)
    if any(f.endswith(('.bin', '.safetensors')) for f in files) or 'config.json' in files:
        model_path = MODEL_PATH
        print(f"Found model at: {MODEL_PATH}")

# Try alternative paths
if model_path is None:
    print(f"Model not found at {MODEL_PATH}")
    print("Trying alternatives...")
    
    for alt_path in ALTERNATIVE_PATHS:
        if os.path.exists(alt_path):
            files = os.listdir(alt_path)
            if any(f.endswith(('.bin', '.safetensors')) for f in files) or 'config.json' in files:
                model_path = alt_path
                print(f"Found at: {alt_path}")
                break
            # Check subdirectories
            for subdir in files:
                subpath = os.path.join(alt_path, subdir)
                if os.path.isdir(subpath):
                    subfiles = os.listdir(subpath)
                    if any(f.endswith(('.bin', '.safetensors')) for f in subfiles):
                        model_path = subpath
                        print(f"Found at: {subpath}")
                        break
        if model_path:
            break

if model_path is None:
    print("\nModel not found!")
    print("\nAvailable in /kaggle/input/:")
    if os.path.exists("/kaggle/input"):
        for item in os.listdir("/kaggle/input"):
            print(f"  📁 {item}")
            subpath = f"/kaggle/input/{item}"
            if os.path.isdir(subpath):
                for sub in os.listdir(subpath)[:5]:
                    print(f"      - {sub}")
    raise FileNotFoundError("Model not found!")

# List model files
print(f"\nModel files:")
for f in os.listdir(model_path)[:10]:
    size = os.path.getsize(os.path.join(model_path, f)) / 1e6
    print(f"{f}: {size:.1f} MB")

Found model at: /kaggle/input/akkadian-pretrained-and-finetuned-model/best

Model files:
config.json: 0.0 MB
spiece.model: 4.3 MB
training_args.bin: 0.0 MB
tokenizer.json: 16.3 MB
tokenizer_config.json: 0.0 MB
model.safetensors: 1200.7 MB
special_tokens_map.json: 0.0 MB
generation_config.json: 0.0 MB


In [4]:


# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
print(f"Tokenizer loaded")

# Load model
print("\nLoading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model = model.to(DEVICE)
model.eval()

# Model info
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Parameters: {n_params:,}")

# Load training info if available
try:
    info_path = os.path.join(model_path, "training_info.json")
    if os.path.exists(info_path):
        with open(info_path, 'r') as f:
            info = json.load(f)
        print(f"\nTraining info:")
        for k, v in info.items():
            print(f"   {k}: {v}")
except:
    pass

# Check model type for prefix
MODEL_TYPE = model.config.model_type
print(f"\nModel type: {MODEL_TYPE}")


Loading tokenizer...
Tokenizer loaded

Loading model...


2026-01-01 04:33:39.645572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767242020.056802      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767242020.175772      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767242021.223374      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767242021.223439      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767242021.223442      17 computation_placer.cc:177] computation placer alr

Model loaded
Parameters: 300,176,768

Model type: mt5


In [5]:
def preprocess_akkadian(text):
    """Clean Akkadian transliteration."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    
    # Standardize breaks
    text = re.sub(r'\[\.{3,}\]|\[…\s*…\]|\.{4,}', ' [GAP] ', text)
    text = re.sub(r'x{3,}', ' [GAP] ', text)
    text = re.sub(r'\bx\b', ' [BREAK] ', text)
    text = re.sub(r'\.{3}|…', ' [BREAK] ', text)
    
    # Remove scribal notations
    text = re.sub(r'[!?]', '', text)
    text = re.sub(r'[/:]', ' ', text)
    text = re.sub(r'[˹˺⌈⌉]', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("Preprocessing function defined!")

Preprocessing function defined!


In [6]:
def translate_single(text):
    """Translate a single Akkadian text."""
    processed = preprocess_akkadian(text)
    
    if len(processed) == 0:
        return ""
    
    # Add prefix for mT5
    if "mt5" in MODEL_TYPE.lower() or "t5" in MODEL_TYPE.lower():
        processed = "translate Akkadian to English: " + processed
    
    # Tokenize
    inputs = tokenizer(
        processed,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True
    ).to(DEVICE)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            early_stopping=True,
        )
    
    # Decode
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation


def translate_batch(texts, batch_size=8, show_progress=True):
    """Translate a batch of texts efficiently."""
    results = []
    total = len(texts)
    
    iterator = range(0, total, batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Translating", total=(total + batch_size - 1) // batch_size)
    
    for i in iterator:
        batch_texts = texts[i:i+batch_size]
        
        # Preprocess
        processed = [preprocess_akkadian(t) for t in batch_texts]
        
        # Track empty inputs
        valid_indices = [j for j, p in enumerate(processed) if len(p) > 0]
        valid_processed = [processed[j] for j in valid_indices]
        
        # Handle all empty
        if len(valid_processed) == 0:
            results.extend([""] * len(batch_texts))
            continue
        
        # Add prefix for mT5
        if "mt5" in MODEL_TYPE.lower() or "t5" in MODEL_TYPE.lower():
            valid_processed = ["translate Akkadian to English: " + p for p in valid_processed]
        
        # Tokenize
        inputs = tokenizer(
            valid_processed,
            return_tensors="pt",
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True
        ).to(DEVICE)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LENGTH,
                num_beams=NUM_BEAMS,
                length_penalty=LENGTH_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM,
                early_stopping=True,
            )
        
        # Decode
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Map back to original positions
        batch_results = [""] * len(batch_texts)
        for j, trans in zip(valid_indices, translations):
            batch_results[j] = trans
        
        results.extend(batch_results)
    
    return results

print("Translation functions defined!")

Translation functions defined!


In [7]:


test_examples = [
    "KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM",
    "a-na 5 GÍN KÙ.BABBAR",
    "um-ma kà-ru-um kà-ni-iš-ma",
    "1 ma-na KÙ.BABBAR",
]

print("\nSample translations:")
for text in test_examples:
    translation = translate_single(text)
    print(f"\n   Input:  {text}")
    print(f"   Output: {translation}")

print("\nModel working!")


Sample translations:

   Input:  KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM
   Output: Seal of Mannum-balum-Aššur son of Ṣill-Adad.

   Input:  a-na 5 GÍN KÙ.BABBAR
   Output: For 5 shekels of silver

   Input:  um-ma kà-ru-um kà-ni-iš-ma
   Output: Thus Kanesh, say to the Kanesh colony:

   Input:  1 ma-na KÙ.BABBAR
   Output: 1 mina of silver

Model working!


In [8]:
test_path = os.path.join(COMPETITION_DATA, "test.csv")
test_df = pd.read_csv(test_path)

print(f"\nTest samples: {len(test_df)}")
print(f"Columns: {test_df.columns.tolist()}")
print(f"\nSample data:")
print(test_df.head(3))


Test samples: 4
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']

Sample data:
   id   text_id  line_start  line_end  \
0   0  332fda50           1         7   
1   1  332fda50           7        14   
2   2  332fda50          14        24   

                                     transliteration  
0  um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...  
1  i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...  
2  ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...  


In [9]:
# Get texts
texts = test_df['transliteration'].tolist()
print(f"\n Total: {len(texts)} texts")
print(f"Batch size: {BATCH_SIZE}")
print(f"Beam search: {NUM_BEAMS}")

# Generate
print("\nGenerating...")
translations = translate_batch(texts, batch_size=BATCH_SIZE, show_progress=True)

print(f"\nGenerated {len(translations)} translations")

# Check for empty translations
empty_count = sum(1 for t in translations if len(t.strip()) == 0)
if empty_count > 0:
    print(f"Warning: {empty_count} empty translations")


 Total: 4 texts
Batch size: 8
Beam search: 5

Generating...


Translating: 100%|██████████| 1/1 [00:23<00:00, 23.43s/it]


Generated 4 translations


In [10]:
# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': translations
})

# Handle any NaN values
submission['translation'] = submission['translation'].fillna('')

# Save
submission.to_csv('submission.csv', index=False)

print(f"\nSaved: submission.csv")
print(f"Shape: {submission.shape}")
print(f"Columns: {submission.columns.tolist()}")


Saved: submission.csv
Shape: (4, 2)
Columns: ['id', 'translation']


In [11]:

# Compare with sample submission
sample_path = os.path.join(COMPETITION_DATA, "sample_submission.csv")
if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    
    print(f"\nComparison:")
    print(f"Sample columns: {sample.columns.tolist()}")
    print(f"Your columns:   {submission.columns.tolist()}")
    
    cols_match = list(sample.columns) == list(submission.columns)
    print(f"Columns match: {'OK' if cols_match else 'No'}")
    
    print(f"\nSample rows: {len(sample)}")
    print(f"Your rows:   {len(submission)}")

# Show sample
print(f"\nSample predictions:")
for i in range(min(5, len(submission))):
    text_id = submission.iloc[i]['id']
    trans = submission.iloc[i]['translation']
    print(f"\nID {text_id}:")
    print(f"{trans[:100]}{'...' if len(trans) > 100 else ''}")


Comparison:
Sample columns: ['id', 'translation']
Your columns:   ['id', 'translation']
Columns match: OK

Sample rows: 4
Your rows:   4

Sample predictions:

ID 0:
Thus Kanesh, say to Aqul-ṭāb, our messenger, and the wabartum: The messenger went to the City.

ID 1:
In accordance with the verdict of the City they must not raise claim against Dān-Aššur. The Kanesh c...

ID 2:
When we hear our missive, you shall come here and come here to the palace. When they come here they ...

ID 3:
The judges seized us against the Kanesh colony and they gave us for these proceedings and we gave ou...


In [12]:
# File check
if os.path.exists('submission.csv'):
    size = os.path.getsize('submission.csv') / 1024
    print(f"\nsubmission.csv: {size:.1f} KB")
    
    # Read back and verify
    check = pd.read_csv('submission.csv')
    print(f"Rows: {len(check)}")
    print(f"Columns: {check.columns.tolist()}")
    
    # Stats
    avg_len = check['translation'].str.len().mean()
    print(f"Avg translation length: {avg_len:.1f} chars")




submission.csv: 0.7 KB
Rows: 4
Columns: ['id', 'translation']
Avg translation length: 170.2 chars
